In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [3]:
df= pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\Air_Quality_Index(AQI)_Projrct\01_Data\raw\city_day.csv\city_day.csv")
df.head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


In [53]:
clean = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\Air_Quality_Index(AQI)_Projrct\01_Data\raw\city_day.csv\city_day.csv")
CITIES = ["Delhi", "Lucknow","Hyderabad","Bhopal","Chennai","Bengaluru"]

print(clean.shape)
print(clean["City"].value_counts())

(29531, 16)
City
Ahmedabad             2009
Bengaluru             2009
Chennai               2009
Mumbai                2009
Lucknow               2009
Delhi                 2009
Hyderabad             2006
Patna                 1858
Gurugram              1679
Visakhapatnam         1462
Amritsar              1221
Jorapokhar            1169
Jaipur                1114
Thiruvananthapuram    1112
Amaravati              951
Brajrajnagar           938
Talcher                925
Kolkata                814
Guwahati               502
Coimbatore             386
Shillong               310
Chandigarh             304
Bhopal                 289
Kochi                  162
Ernakulam              162
Aizawl                 113
Name: count, dtype: int64


In [54]:
df = clean.copy()                       # clean abhi raw data hai, isliye df me copy
df["Date"] = pd.to_datetime(df["Date"])
print(df["Date"].dtype)  

datetime64[ns]


##### Step 2: Select 5 cities
The dataset has 26 cities. We keep 5 that differ in climate and pollution level:
Delhi, Lucknow, Kolkata, Bhopal and Mumbai.

In [55]:
df = df[df["City"].isin(CITIES)].copy()
print("After selecting cities:", df.shape)
print(df["City"].value_counts())

After selecting cities: (10331, 16)
City
Bengaluru    2009
Chennai      2009
Lucknow      2009
Delhi        2009
Hyderabad    2006
Bhopal        289
Name: count, dtype: int64


##### Step 3: Remove duplicate rows
If the same city has two rows for the same date, the model would learn that day twice.
We keep only one row per city per date.

In [56]:
print("Duplicates found:", df.duplicated(subset=["City", "Date"]).sum())
df = df.drop_duplicates(subset=["City", "Date"])

Duplicates found: 0


#### ## Step 4 and 5: Fix dates and fill missing values
Done **for each city separately** (we never mix two cities).
1. Sort by date so the days are in order.
2. Add missing days (`asfreq("D")`). ARIMA and Prophet need one row for every day.
3. Fill empty values with interpolation (a straight line between the last and next known value).
4. `limit=7`: gaps longer than a week are left empty, because guessing that much is unreliable.

In [57]:
pollutants = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "AQI"]
parts = []

for city, g in df.groupby("City"):
    g = g.sort_values("Date").set_index("Date").asfreq("D")
    g["City"] = city
    g[pollutants] = g[pollutants].interpolate(method="time", limit=7)
    parts.append(g.reset_index())

clean = pd.concat(parts, ignore_index=True)
print("After filling gaps:", clean.shape)

After filling gaps: (10331, 16)


##### Step 6: Remove unusable rows
- AQI is our target (the value we predict). A row without AQI is of no use, so we drop it.
- AQI can never be negative. Negative values are sensor errors.

In [58]:
print("Rows with no AQI:", clean["AQI"].isnull().sum())
clean = clean.dropna(subset=["AQI"])

print("Negative AQI values:", (clean["AQI"] < 0).sum())
clean = clean[clean["AQI"] >= 0]

print("Max AQI:", clean["AQI"].max())
print("Shape now:", clean.shape)

Rows with no AQI: 367
Negative AQI values: 0
Max AQI: 737.0
Shape now: (9964, 16)


#### Step 7: Create new columns (feature engineering)
- **AQI_Category:** turns the AQI number into a label (Good, Poor, Severe...). Needed for warnings and health advice.
- **Year, Month, Season:** help us study the yearly pattern and explain why AQI rises.

In [59]:
def aqi_category(v):
    if v <= 50:  return "Good"
    if v <= 100: return "Satisfactory"
    if v <= 200: return "Moderate"
    if v <= 300: return "Poor"
    if v <= 400: return "Very Poor"
    return "Severe"

def season(m):
    if m in (12, 1, 2):   return "Winter"
    if m in (3, 4, 5):    return "Summer"
    if m in (6, 7, 8, 9): return "Monsoon"
    return "Post-monsoon"

clean["AQI_Category"] = clean["AQI"].apply(aqi_category)
clean["Year"]   = clean["Date"].dt.year
clean["Month"]  = clean["Date"].dt.month
clean["Season"] = clean["Month"].apply(season)
clean.head()

,Date,City,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket,AQI_Category,Year,Month,Season
79,2015-03-21,Bengaluru,48.59,NaN,3.47,27.00,18.04,28.02,3.89,1.94,52.96,21.33,196.72,NaN,91.0,Satisfactory,Satisfactory,2015,3,Summer
80,2015-03-22,Bengaluru,47.38,NaN,2.84,22.39,15.33,23.19,11.29,2.05,71.85,14.97,138.20,NaN,120.0,Moderate,Moderate,2015,3,Summer
81,2015-03-23,Bengaluru,65.65,NaN,3.10,26.35,17.45,27.76,9.95,6.30,72.84,9.88,100.77,NaN,154.0,Moderate,Moderate,2015,3,Summer
82,2015-03-24,Bengaluru,60.47,NaN,5.39,29.87,20.88,35.10,1.46,6.07,64.12,5.90,61.48,NaN,119.0,Moderate,Moderate,2015,3,Summer
83,2015-03-25,Bengaluru,62.56,NaN,3.16,23.57,16.39,27.13,10.05,4.98,82.34,4.53,39.99,NaN,232.0,Poor,Poor,2015,3,Summer


#### Step 8: Save the clean data
The clean file goes into the `processed` folder. The raw file is never changed.
All later notebooks will read this clean file.

In [60]:
os.makedirs("../01_Data/processed", exist_ok=True)
clean.to_csv("../01_Data/processed/cleaned_data.csv", index=False)

print("Saved! Final shape:", clean.shape)
print(clean["City"].value_counts())

Saved! Final shape: (9964, 20)
City
Delhi        2009
Bengaluru    1930
Lucknow      1920
Chennai      1913
Hyderabad    1907
Bhopal        285
Name: count, dtype: int64


In [61]:
import os
print("File bani?", os.listdir("../01_Data/processed"))

summary = clean.groupby("City").agg(
    days=("Date", "count"),
    start=("Date", "min"),
    end=("Date", "max"),
    avg_aqi=("AQI", "mean"),
).round(0)
summary

File bani? ['cleaned_data.csv']


,days,start,end,avg_aqi
City,,,,
Bengaluru,1930,2015-03-21,2020-07-01,94.0
Bhopal,285,2019-09-18,2020-07-01,131.0
Chennai,1913,2015-03-24,2020-07-01,115.0
Delhi,2009,2015-01-01,2020-07-01,259.0
Hyderabad,1907,2015-03-31,2020-07-01,109.0
Lucknow,1920,2015-03-21,2020-07-01,217.0


In [62]:
raw = pd.read_csv(RAW, parse_dates=["Date"])
raw = raw.dropna(subset=["AQI"])

ranking = raw.groupby("City").agg(
    days=("Date", "count"),
    start=("Date", "min"),
).sort_values("days", ascending=False)
ranking

,days,start
City,,
Delhi,1999,2015-01-01
Bengaluru,1910,2015-03-21
Lucknow,1893,2015-03-21
Chennai,1884,2015-03-24
Hyderabad,1880,2015-03-31
Patna,1459,2015-10-03
Gurugram,1453,2016-01-23
Ahmedabad,1334,2015-01-29
Visakhapatnam,1171,2016-07-07


In [63]:
RAW = "../01_Data/raw/city_day.csv"
if os.path.isdir(RAW):
    RAW = RAW + "/city_day.csv"

In [64]:
clean.groupby("City").agg(
    days=("Date", "count"),
    start=("Date", "min"),
    avg_aqi=("AQI", "mean"),
).round(0)

,days,start,avg_aqi
City,,,
Bengaluru,1930,2015-03-21,94.0
Bhopal,285,2019-09-18,131.0
Chennai,1913,2015-03-24,115.0
Delhi,2009,2015-01-01,259.0
Hyderabad,1907,2015-03-31,109.0
Lucknow,1920,2015-03-21,217.0
